# EMBER-Dyn Quickstart 02 — XGBoost (template)

**Dataset**: EMBER-Dyn (T1 binary detection)  
**Expected metric**: test ROC AUC in the ~0.99 range; exact value
depends on hyperparameter tuning.  
**Runtime**: ~3 minutes on Colab free CPU tier

Template for users who want to cross-check EMBER-Dyn against a
gradient-boosting baseline other than LightGBM. The paper reports
LightGBM numbers only; use this template to produce an XGBoost row
for your own comparison tables.

In [ ]:
!pip install -q pandas pyarrow scikit-learn xgboost

## Load EMBER-Dyn static features (Zenodo open tier)

In [ ]:
import pandas as pd, numpy as np
import pyarrow.parquet as pq
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

# EMBER-Dyn open tier is archived on Zenodo (DOI in the README). Download it first
# (e.g. `pip install zenodo_get && zenodo_get <ZENODO_DOI>`) and point DATA_DIR at the
# extracted dataset folder.
DATA_DIR = 'dataset/static'
df_train = pq.read_table(f'{DATA_DIR}/train.parquet').to_pandas()
df_val   = pq.read_table(f'{DATA_DIR}/val.parquet').to_pandas()
df_test  = pq.read_table(f'{DATA_DIR}/test.parquet').to_pandas()

feat_cols = [c for c in df_train.columns if c.startswith('feature_')]
def split_X_y(df):
    return df[feat_cols].values.astype('float32'), df['label'].values.astype('int32')
X_train, y_train = split_X_y(df_train)
X_val,   y_val   = split_X_y(df_val)
X_test,  y_test  = split_X_y(df_test)
print(f'train {X_train.shape}  val {X_val.shape}  test {X_test.shape}')

## Train XGBoost (`hist` tree method, 500 rounds, early stopping)

In [ ]:
import xgboost as xgb

model = xgb.XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    tree_method='hist',
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

## Evaluate on the test split

In [ ]:
p_test = model.predict_proba(X_test)[:, 1]
print(f'XGBoost  test ROC AUC = {roc_auc_score(y_test, p_test):.4f}    '
      f'PR AUC = {average_precision_score(y_test, p_test):.4f}    '
      f'F1 = {f1_score(y_test, p_test > 0.5):.4f}')

## Notes

- The paper Table 4 reports LightGBM only (see notebook `01_quickstart_lightgbm.ipynb`).
- For a comparable XGBoost number with the same boosting budget,
  expect test ROC AUC in the ~0.99 range; tuning `max_depth`,
  `min_child_weight`, and `reg_lambda` typically yields the largest
  gains.